In [1]:
import os
import json
import random
from pathlib import Path
from PIL import Image
from unsloth import FastVisionModel
import torch
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

[unsloth.import_fixes|WARNING]Unsloth: torch==2.12.0.dev20260221+cu128 requires torchvision>=0.27.0, but found torchvision==0.26.0.dev20260220+cu128. Please refer to https://pytorch.org/get-started/previous-versions/ for more information.
Detected a pre-release build. Continuing with a warning. Set UNSLOTH_SKIP_TORCHVISION_CHECK=1 to silence this.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\prasasnna\Miniconda3\envs\blackwell\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0301 22:15:02.562000 12684 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!


[unsloth_zoo.log|WARNING]Unsloth: Warning - regex did not match, patch may have failed


In [8]:
MODEL_NAME       = "sanaX3065/Orvion-vl-3b"       # your existing HF model
DATASET_FILE     = r"datasets/executor/aegis_generated_fixed.jsonl"   # browser-specific only
PERFECT_FILE     = None                             # skip — already in the model

OUTPUT_LORA_DIR  = "./orvion_lora"
OUTPUT_MODEL_DIR = "./orvion_aegis_v2"             # merged weights saved here
LOG_DIR          = "./logs"

# ── Hyperparams ───────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH   = 1024    # covers all records safely (max ~811 tokens)
LORA_RANK        = 16      # same rank keeps adapter lightweight
LORA_ALPHA       = 16      # 1:1 ratio for continued fine-tune — less aggressive
LORA_DROPOUT     = 0.05
BATCH_SIZE       = 1       # safest for VL + images on 12GB
GRAD_ACCUM       = 8       # effective batch = 8
EPOCHS           = 2       # 2 is enough — format already learned
LEARNING_RATE    = 5e-5    # lower than fresh training (2e-4) — preserves existing knowledge
WARMUP_STEPS     = 20
WEIGHT_DECAY     = 0.01
LR_SCHEDULER     = "cosine"
SAVE_STEPS       = 100     # checkpoint every 100 steps — safe to resume if crash
LOGGING_STEPS    = 10
SEED             = 42

True

In [12]:
os.path.exists(r'datasets/executor/generated_screenshots/neg_expand_number_input_wrong_selector_step1_1772382161.png')

True

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# DATASET LOADING
# ─────────────────────────────────────────────────────────────────────────────

def load_dataset(dataset_file: str) -> list:
    records = []
    with open(dataset_file) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    random.seed(SEED)
    random.shuffle(records)

    print(f"  Loaded {len(records)} records from {dataset_file}")
    return records


# ─────────────────────────────────────────────────────────────────────────────
# IMAGE LOADING
# ─────────────────────────────────────────────────────────────────────────────

def load_image(image_path: str) -> Image.Image | None:
    path = Path(r"datasets/executor/"+image_path)
    if not path.exists():
        # Try normalising Windows backslashes to forward slashes
        path = Path(image_path.replace("\\", "/"))
    if not path.exists():
        return None
    try:
        img = Image.open(path).convert("RGB")
        # Cap at 1280x800 — keeps VRAM usage stable
        if img.width > 1280 or img.height > 800:
            img.thumbnail((1280, 800), Image.LANCZOS)
        return img
    except Exception:
        return None

# ─────────────────────────────────────────────────────────────────────────────
# RECORD FORMATTING
# ─────────────────────────────────────────────────────────────────────────────

def format_record(record: dict) -> dict | None:
    """
    Convert raw JSONL record → {"messages": [...], "images": [PIL.Image]}
    Returns None if screenshot is missing — record will be skipped.
    """
    messages = record.get("messages", [])
    images = []
    formatted_messages = []

    for msg in messages:
        role = msg["role"]
        content = msg["content"]

        if isinstance(content, str):
            formatted_messages.append({"role": role, "content": content})
            continue

        parts = []
        for block in content:
            if block.get("type") == "text":
                parts.append({"type": "text", "text": block["text"]})

            elif block.get("type") == "image":
                img = load_image(block.get("image", ""))
                if img is None:
                    return None   # missing screenshot — skip record
                images.append(img)
                parts.append({"type": "image"})

        formatted_messages.append({"role": role, "content": parts})

    if not images:
        return None

    return {"messages": formatted_messages, "images": images}



In [14]:
print("\n" + "="*60)
print("  Orvion-VL Continued Fine-tune")
print(f"  Base:    {MODEL_NAME}")
print(f"  Data:    {DATASET_FILE}")
print(f"  Output:  {OUTPUT_MODEL_DIR}")
print("="*60 + "\n")


  Orvion-VL Continued Fine-tune
  Base:    sanaX3065/Orvion-vl-3b
  Data:    datasets/executor/aegis_generated_fixed.jsonl
  Output:  ./orvion_aegis_v2



In [ ]:
print("Step 1: Loading model from HuggingFace...")


model, tokenizer = FastVisionModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,        # auto — bf16 on 5070 Ti (Blackwell)
    load_in_4bit    = True,        # QLoRA 4-bit
)
print(f"  ✓ Model loaded")

# ── 2. Apply LoRA ──────────────────────────────────────────────
print("Step 2: Applying LoRA adapters...")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r                          = LORA_RANK,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",   # saves ~30% VRAM
    random_state               = SEED,
)

Step 1: Loading model from HuggingFace...
==((====))==  Unsloth 2026.2.1: Fast Qwen2_5_Vl patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 5070 Ti Laptop GPU. Num GPUs = 1. Max memory: 11.94 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0.dev20260221+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"  ✓ Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
print("\nStep 3: Preparing dataset...")
raw = load_dataset(DATASET_FILE)
formatted = []
skipped = 0
for rec in raw:
    result = format_record(rec)
    if result is None:
        skipped += 1
    else:
        formatted.append(result)

print(f"  ✓ {len(formatted)} records ready  ({skipped} skipped — missing screenshots)")

if len(formatted) == 0:
    print("\n❌ No valid records. Make sure generated_screenshots/ is in the same folder.")

from datasets import Dataset
hf_dataset = Dataset.from_list(formatted)

In [ ]:
print("\nStep 4: Setting up trainer...")
Path(LOG_DIR).mkdir(exist_ok=True)
Path(OUTPUT_LORA_DIR).mkdir(exist_ok=True)

In [ ]:
trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    data_collator  = UnslothVisionDataCollator(model, tokenizer),
    train_dataset  = hf_dataset,
    args           = SFTConfig(
        per_device_train_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps  = GRAD_ACCUM,
        warmup_steps                 = WARMUP_STEPS,
        num_train_epochs             = EPOCHS,
        learning_rate                = LEARNING_RATE,
        fp16                         = not is_bf16_supported(),
        bf16                         = is_bf16_supported(),
        logging_steps                = LOGGING_STEPS,
        save_steps                   = SAVE_STEPS,
        save_total_limit             = 2,
        output_dir                   = OUTPUT_LORA_DIR,
        logging_dir                  = LOG_DIR,
        optim                        = "adamw_8bit",
        weight_decay                 = WEIGHT_DECAY,
        lr_scheduler_type            = LR_SCHEDULER,
        seed                         = SEED,
        remove_unused_columns        = False,
        report_to                    = "none",
        dataloader_num_workers       = 0,
        dataset_text_field           = "",
        dataset_kwargs               = {"skip_prepare_dataset": True},
        max_seq_length               = MAX_SEQ_LENGTH,
    ),
)

In [ ]:
# ── 5. Train ───────────────────────────────────────────────────
steps_per_epoch = len(formatted) // (BATCH_SIZE * GRAD_ACCUM)
total_steps     = steps_per_epoch * EPOCHS
print(f"\nStep 5: Training...")
print(f"  Records:         {len(formatted)}")
print(f"  Epochs:          {EPOCHS}")
print(f"  Steps/epoch:     ~{steps_per_epoch}")
print(f"  Total steps:     ~{total_steps}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Learning rate:   {LEARNING_RATE} (continued fine-tune)")
print(f"  Est. time:       ~{total_steps * 4 // 60} min on RTX 5070 Ti\n")

stats = trainer.train()

In [ ]:
# ── 6. Save LoRA ───────────────────────────────────────────────
print("\nStep 6: Saving LoRA adapter...")
model.save_pretrained(OUTPUT_LORA_DIR)
tokenizer.save_pretrained(OUTPUT_LORA_DIR)
print(f"  ✓ LoRA adapter → {OUTPUT_LORA_DIR}")

# ── 7. Merge LoRA into model weights ───────────────────────────
print("\nStep 7: Merging LoRA weights into model (bf16)...")
print(f"  This takes 3-5 minutes and needs ~6GB free VRAM...")
Path(OUTPUT_MODEL_DIR).mkdir(exist_ok=True)

model.save_pretrained_merged(
    OUTPUT_MODEL_DIR,
    tokenizer,
    save_method = "merged_16bit",   # bf16 merged weights — ready to deploy
)
print(f"  ✓ Merged model → {OUTPUT_MODEL_DIR}")

# ── 8. Done ────────────────────────────────────────────────────
runtime_min = stats.metrics.get("train_runtime", 0) / 60
print("\n" + "="*60)
print("  ✅ Done!")
print(f"  Training time:  {runtime_min:.1f} min")
print(f"  Final loss:     {stats.metrics.get('train_loss', 0):.4f}")
print(f"  LoRA adapter:   {OUTPUT_LORA_DIR}")
print(f"  Merged model:   {OUTPUT_MODEL_DIR}")
print("="*60)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model     = AutoModelForCausalLM.from_pretrained("{OUTPUT_MODEL_DIR}")
tokenizer = AutoTokenizer.from_pretrained("{OUTPUT_MODEL_DIR}")

model.push_to_hub("sanaX3065/Orvion-vl-3b-aegis-v2")
tokenizer.push_to_hub("sanaX3065/Orvion-vl-3b-aegis-v2")

In [ ]:
def test_inference(screenshot_path: str, task: str):
    """
    Quick sanity check after training.

    Usage:
        from orvion_train import test_inference
        test_inference("generated_screenshots/my_step.png",
                       "Verify the Login button is visible on the page.")
    """
    from unsloth import FastVisionModel

    print(f"Loading merged model from {OUTPUT_MODEL_DIR}...")
    model, tokenizer = FastVisionModel.from_pretrained(
        OUTPUT_MODEL_DIR,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = None,
        load_in_4bit   = True,
    )
    FastVisionModel.for_inference(model)

    img = Image.open(screenshot_path).convert("RGB")

    system = (
        "You are Aegis. Current Mode: QUALITY_TESTER\n"
        "Tools: [click, type, clear_and_type, open_url, verify_element_visible, "
        "verify_input_value, verify_text_present, verify_url_contains, "
        "raise_bug_ticket, mark_step_pass, mark_flow_blocked]"
    )
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system}]},
        {"role": "user",   "content": [{"type": "image"}, {"type": "text", "text": task}]},
    ]

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs     = tokenizer(img, input_text, return_tensors="pt").to("cuda")

    import torch
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens = 256,
            temperature    = 0.1,
            do_sample      = True,
        )

    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("\n--- Model Response ---")
    print(response)
    return response